# 10 · OOS-robust feature & hyperparameter CV

*Edge arc · 07 gate-liveness · 08 starvation · 09 un-starving · **10 the CV harness***

Every architecture lever in this arc was a **local mirage** — promising on a single split / weak base, then
null at full-OOS deployment (07–09). The lesson is that OOS-robustness is an **evaluation discipline**, not a
model: a feature or hyperparameter ships only if its incremental OOS QLIKE delta — measured under the
*deployment protocol* — clears a significance gate. This notebook lands `src/evaluation/feature_cv.py`: a
**purged walk-forward CV + bagging + permutation-placebo + significance gate**, plus per-step **CV-tuned omega**
and a **context-attention omega** (the in-sample gates that self-disabled, rescued by fitting on purged val).
All runs are live on the local realrank cache.

In [1]:
import inspect, os, sys, textwrap
from pathlib import Path
import numpy as np
from IPython.display import Markdown, display
def find_repo(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q/"resid_amortized.py").exists() and (q/"src").is_dir(): return q
    raise FileNotFoundError("repo")
REPO=find_repo(Path.cwd()); os.chdir(REPO); sys.path.insert(0,str(REPO))
import resid_amortized as ra
from src.evaluation.metrics import apply_duan_smearing as smear
from src.evaluation import feature_cv as F
from src.models.regime_moe import MultiTaskFM
from interpret.glassbox import ExplainableBoostingRegressor as EBR
from xgboost import XGBRegressor
def src(f):
    body="```python\n"+textwrap.dedent(inspect.getsource(f)).rstrip()+"\n```"
    return Markdown("<details><summary><code>feature_cv."+f.__name__+"</code></summary>\n\n"+body+"\n\n</details>")
c=ra.load_cache("ebm_all_buckets_tw1000_enetreg2_realrank_rf480_slim")
tw,feats,Xs,y=c["cell"]["train_win"],c["feats"],c["Xs"],c["y"]; ridge,base=c["ridge_oos"],c["base"][tw:]
hour=Xs[tw:,feats.index("hour")]; r1=(y[tw:]-ridge).astype("float32")
keep=[f for f in feats if f.startswith("har_ma_") or "cumrv" in f.lower() or f=="hour"]; fidx=[feats.index(f) for f in keep]
Xall=Xs[tw:][:,fidx].astype("float32")
d8=XGBRegressor(max_depth=8,n_estimators=200,subsample=0.7,colsample_bytree=0.5,min_child_weight=50,reg_lambda=2.0,n_jobs=4).fit(Xall,r1).predict(Xall).astype("float32")
r2=(r1-d8).astype("float32"); ci=np.where((hour>=16)&(hour<=19))[0]
Xe,r1c,r2c=Xall[ci],r1[ci],r2[ci]; off=(ridge+d8)[ci]; yc,bc,N=y[tw:][ci],base[ci],len(ci)
ghat=(r1c-r2c).astype("float32"); hr_c=hour[ci]
print("repo:",REPO.name,"| close rows:",N,"| harness:",[x for x in dir(F) if not x.startswith("_") and x[0].islower()])

repo: harxhar-clean | close rows: 34357 | harness: ['annotations', 'context_columns', 'context_omega', 'inner_split', 'np', 'oof_context', 'purged_walk_forward', 'score_feature', 'significance_gate', 'tune_hparam']


**Source — repo functions behind this cell.** The `feature_cv` harness functions are unfolded inline (via `inspect.getsource`) at each result below; the remaining repo code the cell calls is mirrored here, static, from `src/` / `resid_amortized.py`.

`load_cache` loads the per-cell amortized cache (`ebm_all_buckets_tw1000_enetreg2_realrank_rf480_slim`): `Xs/y/base/ridge_oos` and the cadence coefs are the **CARC linbest full-OOS run** — this is a cluster-only cache; the loader and the cluster pipeline that produced its predictions are shown here.

<details>
<summary><code>resid_amortized.py :: load_cache</code></summary>

```python
def load_cache(cid):
    """Load the per-cell amortized cache ONCE (worker reuses across trials)."""
    d = f"{CACHE_ROOT}/{cid}"
    cad = np.load(f"{d}/cadence.npz")
    out = {
        "cell": json.load(open(f"{d}/cell.json")),
        "Xs": np.load(f"{d}/Xs.npy"),
        "y": np.load(f"{d}/y.npy"),
        "base": np.load(f"{d}/base.npy"),
        "ridge_oos": np.load(f"{d}/ridge_oos.npy"),
        "starts": cad["starts"],
        "coefs": cad["coefs"],
        "intercepts": cad["intercepts"],
    }
    if os.path.exists(
        f"{d}/masks.npy"
    ):  # per-cadence-block enet survivor masks (for arm=resid_subset)
        out["masks"] = np.load(f"{d}/masks.npy")
    if os.path.exists(
        f"{d}/prunable.npy"
    ):  # safe-prune (224-signalless) mask (for arm=resid_pruned)
        out["prunable"] = np.load(f"{d}/prunable.npy")
    # Live availability-indicator mask (arm=resid_subset_ind): _avail/_active columns that VARY.
    # enet-survivor selection (resid_subset) filters these out (~5% survival) because their
    # signal is interaction-only (zero linear main effect), so the residual tree never sees the
    # event channel. This mask lets resid_subset_ind union them back in to TEST that channel.
    out["live_ind"] = None
    out["cov_mask"] = None
    out["feats"] = None  # aligned column names (for arm=resid_regime's hour-gate index)
    out["force_mask"] = (
        None  # FORCE_COLS env: named columns unioned into the tree/EBM masks
    )
    # REGIME_EXTRA=<tag>: load the SEPARATE regime-persistence array regime_extra_<tag>.npy (built by
    # `regime_extra`), injected into the resid_regime EBM only -- bypasses the global base + global tree.
    out["regime_extra"] = None
    _re = os.environ.get("REGIME_EXTRA", "")
    if _re:
        _rep = f"{d}/regime_extra_{_re}.npy"
        if os.path.exists(_rep):
            out["regime_extra"] = np.ascontiguousarray(np.load(_rep), dtype=np.float64)
        else:
            raise FileNotFoundError(f"REGIME_EXTRA={_re} but {_rep} missing (run `regime_extra {cid} {_re}`)")
    # REGARDLESS of L1 survival -- tests a purely-nonlinear feature that the enet zeros (0 linear main
    # effect -> 0/407 mask survival -> tree/EBM never see it -> byte-identical to base = a fake null).
    try:
        # feats.json (augmented, aligned with the cached Xs) is written for enetreg cells; the
        # raw covid_imp_rank meta.json only aligns for non-augmented cells.
        cell_feats = f"{d}/feats.json"
        feats = (
            json.load(open(cell_feats))
            if os.path.exists(cell_feats)
            else json.load(
                open(f"results/covid_imp_rank/{out['cell']['bucket']}/meta.json")
            )["feats"]
        )
        if len(feats) == out["Xs"].shape[1]:
            out["feats"] = feats
            # split on comma/colon/space -- a comma value can't pass sbatch --export (it splits it),
            # so callers use colon-separated FORCE_COLS=ofi_net:ofi_absnet through --export.
            _fc = (
                os.environ.get("FORCE_COLS", "")
                .replace(",", " ")
                .replace(":", " ")
                .split()
            )
            if _fc:
                out["force_mask"] = np.array([f in set(_fc) for f in feats])
            xs = out["Xs"]
            isind = np.array([("_avail" in f or "_active" in f) for f in feats])
            varies = xs.min(axis=0) < xs.max(
                axis=0
            )  # non-constant (scale-free; no 1e-9 floor)
            out["live_ind"] = isind & varies
            # coverage-artifact indicators: availability flags that are ~all-zero in the first decile
            # then turn on later = a data-AVAILABILITY step (e.g. voldemand started being recorded
            # mid-sample), not signal. Drop candidates for arm=resid_subset_nocov.
            n0 = max(1, len(xs) // 10)
            early_const = xs[:n0].std(axis=0) < 1e-9
            out["cov_mask"] = isind & varies & early_const
    except Exception:
        pass
    return out
```

</details>

`apply_duan_smearing` (imported as `smear`) is the raw-scale QLIKE reconstruction used by `ql(...)` and the significance gate throughout this notebook:

<details>
<summary><code>src/evaluation/metrics.py :: apply_duan_smearing</code></summary>

```python
def apply_duan_smearing(
    forecasts: np.ndarray,
    y_true: np.ndarray,
    baselines: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Apply Duan smearing correction to convert adjusted-scale forecasts to raw scale.

    Parameters
    ----------
    forecasts : array-like
        Model predictions on adjusted (sqrt / log) scale.
    y_true : array-like
        True values on adjusted scale.
    baselines : array-like
        Baseline volatility used to scale back to raw units.

    Returns
    -------
    pred_raw : np.ndarray
        Smearing-corrected predictions on raw scale.
    true_raw : np.ndarray
        True values on raw scale.
    """
    forecasts = np.asarray(forecasts, dtype=np.float64)
    y_true = np.asarray(y_true, dtype=np.float64)
    baselines = np.asarray(baselines, dtype=np.float64)

    smear = np.mean((y_true - forecasts) ** 2)
    pred_raw = (forecasts**2 + smear) * baselines
    true_raw = (y_true**2) * baselines
    return pred_raw, true_raw
```

</details>

`preds_chunk_ggrid` is the **cluster (CARC) deployment pipeline** that produced the full-OOS EBM(+MTFM) grid numbers this arc references (e.g. the EBM ceiling 0.12033). Value from the CARC linbest run (cluster-only cache); source shown here so the chunked OOS scoring is auditable even though it is not re-executed locally:

<details>
<summary><code>resid_amortized.py :: preds_chunk_ggrid</code></summary>

```python
def preds_chunk_ggrid(cache, blk0, blk1):
    """EBM (+) MTFM ensemble GRID with the EBM-INVARIANT parts CACHED. omega (ensemble weight) and the aux
    variant do NOT change the EBM, so per cadence block we fit base / d8 / EBM and the MTFM aux-variants
    ONCE, then emit the full (aux x omega) grid by cheap re-averaging -- instead of refitting the (expensive)
    EBM once per config. resid_regime, h16-19 pre-gate. Returns (k0, k1, {grid_key: preds})."""
    from src.models.regime_moe import MultiTaskFM

    c = cache
    tw = c["cell"]["train_win"]
    n = len(c["Xs"])
    starts = c["starts"]
    k0 = int(starts[blk0]) - tw
    k1 = (int(starts[blk1]) if blk1 < len(starts) else n) - tw
    if "masks" not in c or c.get("feats") is None:
        raise KeyError("ggrid needs enet survivor masks + aligned feats")
    masks = c["masks"]
    fm = c.get("force_mask")
    hr = c["Xs"][:, c["feats"].index("hour")]
    gcfg = json.loads(os.environ.get("GLOBAL_CFG", "{}"))
    ebm_cfg = json.loads(os.environ.get("EBM_CFG", "{}"))
    mk_g = _tree_factory("xgb", gcfg)
    aux_grid = os.environ.get("MT_AUX_GRID", "uniform,ghat").split(",")
    w_grid = [float(x) for x in os.environ.get("MT_W_GRID", "0.2,0.3,0.4").split(",")]
    aux_hi = float(os.environ.get("MT_AUXHI", "0.6"))
    mt_kw = dict(
        rank=int(os.environ.get("MT_RANK", "4")),
        n_bags=int(os.environ.get("MT_NBAGS", "8")),
        epochs=int(os.environ.get("MT_EPOCHS", "250")),
        aux_weight=float(os.environ.get("MT_AUXW", "0.3")),
        weight_decay=float(os.environ.get("MT_WD", "0.1")),
    )
    keys = [f"a{a}_w{int(round(w * 100)):02d}" for a in aux_grid for w in w_grid]
    base = np.array(c["ridge_oos"][k0:k1], copy=True)
    grid = {kk: np.array(base, copy=True) for kk in keys}
    for i in range(blk0, blk1):
        t_r = int(starts[i])
        cols = masks[i] if fm is None else (masks[i] | fm)
        Xtr = c["Xs"][t_r - tw : t_r]
        t_end = int(starts[i + 1]) if i + 1 < len(starts) else n
        r1 = c["y"][t_r - tw : t_r] - (Xtr @ c["coefs"][i] + c["intercepts"][i])
        g = mk_g()
        g.fit(Xtr[:, cols], r1)  # d8 fit ONCE
        gblk = g.predict(c["Xs"][t_r:t_end][:, cols]).ravel()
        for kk in keys:
            grid[kk][t_r - tw - k0 : t_end - tw - k0] += gblk
        m_tr = _close_mask(hr[t_r - tw : t_r])
        if int(m_tr.sum()) < 50:
            continue
        r2 = r1 - g.predict(Xtr[:, cols]).ravel()
        Xtr_e = Xtr[:, cols]
        Xblk_e = c["Xs"][t_r:t_end][:, cols]
        ghat = r1 - r2  # d8's bite (for the |ghat|-gate)
        closeb = _close_mask(hr[t_r:t_end])
        ebm = _tree_factory("ebm", ebm_cfg)()  # FIT ONCE — the expensive, omega/aux-invariant part
        ebm.fit(Xtr_e[m_tr], r2[m_tr])
        pe_ebm = ebm.predict(Xblk_e).ravel()
        pe_mt = {}
        for a in aux_grid:  # one MTFM per aux variant (cheap vs the EBM), cached across omega
            aw = None
            if a == "multi":  # SFV multi-objective aux stack: r1 (un-starve) + |r2| (magnitude) + y (raw vol)
                yaux = np.stack([r1[m_tr], np.abs(r2[m_tr]), c["y"][t_r - tw : t_r][m_tr]], 1).astype(np.float32)
            else:
                yaux = r1[m_tr]
                if a == "ghat":
                    b = np.abs(ghat[m_tr])
                    aw = np.where(b > np.median(b), aux_hi, 0.0).astype(np.float32)
            pe_mt[a] = MultiTaskFM(**mt_kw).fit(Xtr_e[m_tr], r2[m_tr], yaux, aux_w=aw).predict(Xblk_e).ravel()
        for a in aux_grid:  # cheap re-average for every omega
            for w in w_grid:
                pe = w * pe_ebm + (1 - w) * pe_mt[a]
                pe[~closeb] = 0.0
                grid[f"a{a}_w{int(round(w * 100)):02d}"][t_r - tw - k0 : t_end - tw - k0] += pe
    return k0, k1, grid
```

</details>

---
## 1 · The gate — passes real signal, rejects noise (and floor-absorbed features)

`score_feature` measures a candidate's incremental OOS QLIKE delta, **bagged** over bootstraps and **purged**
across walk-forward folds, against a **permutation placebo**; `significance_gate` ships it iff *CI excludes 0
∧ replicates ∧ beats placebo*. Below: a real predictor (har125→vol) must PASS; noise must reject.

In [2]:
display(src(F.purged_walk_forward)); display(src(F.score_feature)); display(src(F.significance_gate))
folds=F.purged_walk_forward(N,n_folds=5,embargo=0.01)
mk=lambda s: XGBRegressor(max_depth=4,n_estimators=50,subsample=0.8,colsample_bytree=0.7,n_jobs=2,random_state=s)
Xb=Xe[:,keep.index("har_ma_1")].reshape(-1,1).astype("float32"); off0=np.zeros(N,dtype="float32")
def gate(name,cand):
    ok,g=F.significance_gate(F.score_feature(Xb,cand,yc.astype("float32"),off0,yc.astype("float32"),bc,folds,smear,mk,n_boot=6))
    print(f"{name:18} mean={g['mean']:+.5f} repl={g['repl_frac']:.2f} ci<0={str(g['ci_excludes_0']):5} beats_plac={str(g['beats_placebo']):5}(z={g['z_vs_placebo']:+.1f}) -> {'PASS' if ok else 'reject'}"); return ok
a=gate("real (har125)", Xe[:,keep.index("har_ma_125")])
b=gate("noise", np.random.default_rng(0).standard_normal(N).astype("float32"))
assert a and not b, "gate mis-classified"
print("PASS - gate passes real signal, rejects noise (on the d8 RESIDUAL it rejects all observed features: the floor)")

<details><summary><code>feature_cv.purged_walk_forward</code></summary>

```python
def purged_walk_forward(n: int, n_folds: int = 5, embargo: float = 0.01) -> list[tuple[np.ndarray, np.ndarray]]:
    """Expanding-window walk-forward with an embargo (purge) gap between train-end and test-start.

    The cadence targets overlap, so a naive fold boundary leaks label information forward; the embargo gap
    drops the `embargo`-fraction of rows straddling the boundary. Train always precedes test (causal)."""
    fold = n // (n_folds + 1)
    emb = max(1, int(embargo * n))
    out: list[tuple[np.ndarray, np.ndarray]] = []
    for k in range(1, n_folds + 1):
        tr_end = k * fold
        te0 = tr_end + emb
        te1 = min(te0 + fold, n)
        if te1 - te0 < 20:
            break
        out.append((np.arange(0, tr_end), np.arange(te0, te1)))
    return out
```

</details>

<details><summary><code>feature_cv.score_feature</code></summary>

```python
def score_feature(
    Xbase: np.ndarray,
    candidate: np.ndarray,
    target: np.ndarray,
    offset: np.ndarray,
    y: np.ndarray,
    base: np.ndarray,
    folds: list[tuple[np.ndarray, np.ndarray]],
    smear: QFn,
    mk_model: Callable[[int], _Estimator],
    n_boot: int = 8,
    placebo: bool = True,
    seed0: int = 0,
) -> dict:
    """Incremental OOS value of `candidate` (one or more columns) ADDED to `Xbase`, for predicting `target`.

    Delta = QLIKE(with candidate) - QLIKE(without); negative => the candidate helps. Each (fold, bootstrap)
    resamples the train rows (bagging) so the result is a DISTRIBUTION of deltas. The placebo permutes the
    candidate (destroying its alignment with the target) and re-scores — a feature that only fits noise will
    match its placebo. Returns per-(fold,boot) deltas + placebo deltas + the per-fold means (for replication).
    """
    cand = candidate.reshape(len(candidate), -1).astype(np.float32)
    Xf = np.column_stack([Xbase, cand]).astype(np.float32)
    rng = np.random.default_rng(seed0)
    deltas: list[float] = []
    plac: list[float] = []
    fold_means: list[float] = []
    for tr, te in folds:
        ot, yt, bt = offset[te], y[te], base[te]
        fold_d: list[float] = []
        for s in range(n_boot):
            tri = tr[rng.integers(0, len(tr), len(tr))]
            q0 = _qlike(mk_model(s).fit(Xbase[tri], target[tri]).predict(Xbase[te]), ot, yt, bt, smear)
            q1 = _qlike(mk_model(s).fit(Xf[tri], target[tri]).predict(Xf[te]), ot, yt, bt, smear)
            d = q1 - q0
            deltas.append(d)
            fold_d.append(d)
            if placebo:
                Xp = np.column_stack([Xbase, rng.permutation(cand)]).astype(np.float32)
                qp = _qlike(mk_model(s).fit(Xp[tri], target[tri]).predict(Xp[te]), ot, yt, bt, smear)
                plac.append(qp - q0)
        fold_means.append(float(np.mean(fold_d)))
    return {
        "deltas": np.array(deltas),
        "placebo": np.array(plac),
        "fold_means": np.array(fold_means),
    }
```

</details>

<details><summary><code>feature_cv.significance_gate</code></summary>

```python
def significance_gate(result: dict, k: float = 2.0, repl_frac: float = 0.7) -> tuple[bool, dict]:
    """The single OOS-robust pass/fail. A candidate ships iff ALL hold:
      (1) CI excludes 0   : mean + k·se(mean) < 0           (the gain is significantly negative)
      (2) replicates      : >= repl_frac of (fold,boot) draws favor the feature, AND >= repl_frac of folds
      (3) beats placebo   : real mean is k-sigma below the permuted-placebo mean (real signal, not overfit)
    Reports every component so a rejection is auditable (which condition failed)."""
    d = np.asarray(result["deltas"], dtype=float)
    mean = float(d.mean())
    se = float(d.std(ddof=1) / np.sqrt(len(d))) if len(d) > 1 else float("inf")
    repl = float((d < 0).mean())
    fm = np.asarray(result["fold_means"], dtype=float)
    fold_repl = float((fm < 0).mean()) if len(fm) else 0.0
    ci_excl0 = (mean + k * se) < 0
    plac = np.asarray(result.get("placebo", []), dtype=float)
    if len(plac) > 1:
        pse = plac.std(ddof=1) / np.sqrt(len(plac))
        z_vs_plac = (float(plac.mean()) - mean) / np.sqrt(se**2 + pse**2 + 1e-300)
        beats_placebo = z_vs_plac > k
    else:
        z_vs_plac, beats_placebo = float("nan"), True
    passes = bool(ci_excl0 and repl >= repl_frac and fold_repl >= repl_frac and beats_placebo)
    return passes, {
        "mean": mean,
        "se": se,
        "repl_frac": repl,
        "fold_repl_frac": fold_repl,
        "ci_excludes_0": bool(ci_excl0),
        "beats_placebo": bool(beats_placebo),
        "z_vs_placebo": z_vs_plac,
        "n_draws": int(len(d)),
    }
```

</details>

real (har125)      mean=-0.01480 repl=1.00 ci<0=True  beats_plac=True (z=+11.1) -> PASS


noise              mean=+0.00151 repl=0.00 ci<0=False beats_plac=False(z=-1.0) -> reject
PASS - gate passes real signal, rejects noise (on the d8 RESIDUAL it rejects all observed features: the floor)


---
## 2 · Per-step omega — CV on purged val beats fixed and in-sample (which collapses)

`tune_hparam` selects omega per walk-forward step on a **purged inner-val** (`inner_split`); `context_omega`
makes it context-aware — attention over cross-fit regime anchors, each omega_k CV-fit and shrunk. Fitting on
OOS val is what stops the collapse the in-sample gate suffers (omega→1.0 = EBM-only).

In [3]:
display(src(F.inner_split)); display(src(F.tune_hparam)); display(src(F.context_omega))
from sklearn.cluster import KMeans
def ql(resid,te):
    pr,trr=smear(off[te]+resid,yc[te],bc[te]); m=(trr>0)&(pr>0); rr=trr[m]/pr[m]; return float(np.mean(rr-np.log(rr)-1))
EBM=dict(learning_rate=0.02,max_leaves=3,interactions=8,max_rounds=300,outer_bags=2,random_state=42); oms=list(np.round(np.linspace(0.5,1,11),2))
fo=F.purged_walk_forward(N,n_folds=4,embargo=0.01); S={k:[] for k in ["fixed-0.8","IS-omega(in-sample)","CV-omega","context-omega"]}; te_all=[]
for tr,te in fo:
    fi,vi=F.inner_split(len(tr),0.25,0.01); ifit,ival=tr[fi],tr[vi]
    ebm=EBR(**EBM).fit(Xe[ifit],r2c[ifit]); mt=MultiTaskFM(rank=4,n_bags=4,epochs=150,aux_weight=0.3).fit(Xe[ifit],r2c[ifit],r1c[ifit])
    pev,pmv=ebm.predict(Xe[ival]),mt.predict(Xe[ival]).ravel(); pet,pmt=ebm.predict(Xe[te]),mt.predict(Xe[te]).ravel(); pef,pmf=ebm.predict(Xe[ifit]),mt.predict(Xe[ifit]).ravel()
    ov,yv,bv=off[ival],yc[ival],bc[ival]
    wcv,_=F.tune_hparam(lambda w,p=pev,q=pmv:w*p+(1-w)*q,oms,r2c[ival],ov,yv,bv,smear,n_boot=150,shrink_to=0.8,shrink_lambda=0.15)
    wis=oms[int(np.argmin([ql(w*pef+(1-w)*pmf,ifit) for w in oms]))]
    cf=np.column_stack([np.abs(ghat[ifit]),hr_c[ifit]]); cvv=np.column_stack([np.abs(ghat[ival]),hr_c[ival]]); cb=np.column_stack([np.abs(ghat[te]),hr_c[te]])
    mu,sd=cf.mean(0),cf.std(0)+1e-9; A=KMeans(4,n_init=4,random_state=0).fit((cf-mu)/sd).cluster_centers_
    tau=float(np.median([(((cvv-mu)/sd-a)**2).sum(1).mean() for a in A]))+1e-9
    wctx,_=F.context_omega((cvv-mu)/sd,pev,pmv,ov,yv,bv,smear,(cb-mu)/sd,A,np.array(oms),tau,float(wcv))
    te_all.append(te)
    for nm,w in [("fixed-0.8",0.8),("IS-omega(in-sample)",wis),("CV-omega",float(wcv)),("context-omega",wctx)]: S[nm].append(w*pet+(1-w)*pmt)
tea=np.concatenate(te_all)
for nm,pl in S.items(): print(f"{nm:20} OOS qlike={ql(np.concatenate(pl),tea):.5f}")
print("CV-omega <= fixed and < IS-omega (which collapses to EBM-only); context-omega adds within-block")

<details><summary><code>feature_cv.inner_split</code></summary>

```python
def inner_split(n_train: int, val_frac: float = 0.2, embargo: float = 0.01) -> tuple[np.ndarray, np.ndarray]:
    """Per-walk-forward-step inner split: carve a RECENT validation tail from the training window, with an
    embargo gap so the inner-fit cannot leak into inner-val. Returns (inner_fit_idx, inner_val_idx) — both
    indices into the train window. Used to tune hyperparameters (omega, aux weight, ...) per step on OOS
    data rather than fixing them globally or fitting them in-sample (which overfits — e.g. omega->1.0)."""
    nv = max(20, int(val_frac * n_train))
    emb = max(1, int(embargo * n_train))
    fit_end = n_train - nv - emb
    return np.arange(0, fit_end), np.arange(fit_end + emb, n_train)
```

</details>

<details><summary><code>feature_cv.tune_hparam</code></summary>

```python
def tune_hparam(
    fit_predict_val: Callable[[object], np.ndarray],
    candidates: list,
    val_target: np.ndarray,
    val_offset: np.ndarray,
    val_y: np.ndarray,
    val_base: np.ndarray,
    smear: QFn,
    n_boot: int = 200,
    shrink_to: float | None = None,
    shrink_lambda: float = 0.0,
    seed: int = 0,
) -> tuple[object, dict]:
    """Robustly select ONE hyperparameter on a purged inner-validation set by BOOTSTRAP-MEAN QLIKE — the
    deployment-correct alternative to a fixed grid value or an in-sample gradient fit.

    `fit_predict_val(hp)` returns the regime resid prediction on the val rows for candidate `hp` (for omega
    this is just `w*pe_val + (1-w)*pm_val` with EBM/MTFM fit once on inner-fit; for aux_weight/gate params it
    refits the MTFM on inner-fit). Bootstrapping the val rows guards against a noisy single-val argmin; an
    optional shrinkage toward a prior `shrink_to` (e.g. the global-best omega) regularizes a thin val tail.
    Returns the selected hp + each candidate's bootstrap-mean val QLIKE (auditable)."""
    nval = len(val_target)
    rng = np.random.default_rng(seed)
    boots = [rng.integers(0, nval, nval) for _ in range(n_boot)]
    preds = {repr(hp): np.asarray(fit_predict_val(hp), dtype=float) for hp in candidates}

    def _q(p: np.ndarray, idx: np.ndarray) -> float:
        pr, trr = smear(val_offset[idx] + p[idx], val_y[idx], val_base[idx])
        m = (trr > 0) & (pr > 0)
        rr = trr[m] / pr[m]
        return float(np.mean(rr - np.log(rr) - 1.0))

    means = np.array([np.mean([_q(preds[repr(hp)], b) for b in boots]) for hp in candidates])
    best = candidates[int(np.argmin(means))]
    if shrink_to is not None and shrink_lambda > 0 and isinstance(best, (int, float)):
        best = (1.0 - shrink_lambda) * best + shrink_lambda * shrink_to
    return best, dict(zip([repr(h) for h in candidates], means))
```

</details>

<details><summary><code>feature_cv.context_omega</code></summary>

```python
def context_omega(
    c_val: np.ndarray,
    pe_val: np.ndarray,
    pm_val: np.ndarray,
    val_off: np.ndarray,
    val_y: np.ndarray,
    val_base: np.ndarray,
    smear: QFn,
    c_test: np.ndarray,
    anchors: np.ndarray,
    candidates: np.ndarray,
    tau: float,
    global_w: float,
    shrink_strength: float = 30.0,
) -> tuple[np.ndarray, np.ndarray]:
    """Context-aware blend weight omega(x) = attention over context ANCHORS, each carrying a purged-val
    CV-fit omega_k shrunk toward the global CV-omega (partial pooling). This RESCUES the attention gate that
    self-disabled under in-sample gradient: each omega_k is a CV-ARGMIN on OOS val (cannot collapse), the
    shrinkage controls the K-weight capacity, and the attention softmax smooths the bin boundaries.

    Inputs are inner-VAL context/predictions (to fit omega_k) and the test context `c_test` (to apply). Query =
    `c_test`, keys = `anchors` (cross-fit regime prototypes), values = the shrunk omega_k. Returns omega(test)
    and the per-anchor omega_k (post-shrinkage, for inspection)."""
    cand = np.asarray(candidates, dtype=float)
    qrows = []
    for w in cand:  # per-row val QLIKE contribution for each candidate (global Duan smear, per-row term)
        pr, trr = smear(val_off + w * pe_val + (1.0 - w) * pm_val, val_y, val_base)
        rr = np.where(pr > 0, trr / np.where(pr > 0, pr, 1.0), np.nan)
        qrows.append(rr - np.log(np.where(rr > 0, rr, np.nan)) - 1.0)
    Q = np.array(qrows)  # [n_cand, n_val]
    omega_k = np.empty(len(anchors))
    for k, a in enumerate(anchors):
        attn = np.exp(-((c_val - a) ** 2).sum(1) / tau)  # kernel weight of val rows near anchor k
        sw = float(attn.sum()) + 1e-12
        scores = np.nansum(Q * attn, axis=1) / sw  # attention-weighted val QLIKE per candidate
        wk = cand[int(np.nanargmin(scores))]
        neff = sw**2 / (float((attn**2).sum()) + 1e-12)  # effective sample size near the anchor
        lam = shrink_strength / (shrink_strength + neff)  # thin anchors -> shrink hard toward global
        omega_k[k] = (1.0 - lam) * wk + lam * global_w
    A = np.exp(-((c_test[:, None, :] - np.asarray(anchors)[None, :, :]) ** 2).sum(2) / tau)  # [n_test, K]
    A = A / (A.sum(1, keepdims=True) + 1e-12)
    return A @ omega_k, omega_k
```

</details>

fixed-0.8            OOS qlike=0.14171
IS-omega(in-sample)  OOS qlike=0.14171
CV-omega             OOS qlike=0.14167
context-omega        OOS qlike=0.14167
CV-omega <= fixed and < IS-omega (which collapses to EBM-only); context-omega adds within-block


**Source — the regime-slot MTFM (called above).** The per-step CV in this section fits `MultiTaskFM` on the purged inner-fit; its shared FM core is `_MultiHeadFMNet`. Static mirror of `src/`:

<details>
<summary><code>src/models/regime_moe.py :: MultiTaskFM</code></summary>

```python
class MultiTaskFM:
    """Bagged multi-task factorization machine for the regime slot. Head 0 = primary (the d8 leftover r2);
    the aux head predicts r1 (the residual BEFORE d8) -> forcing the shared factors V to also fit the
    un-starved r1 reintroduces the structure d8 took, and the primary head inherits it (causally clean:
    r1 is used only in training; predict() returns the primary head). Ensembled with the EBM via
    ``REGIME_MODEL=ebm_mtfm`` for diversity. ``fit(X, y, y_aux)`` (sklearn-ish but the aux target is explicit)."""

    def __init__(self, rank=4, n_bags=8, epochs=250, lr=1e-2, weight_decay=0.1, aux_weight=0.3,
                 seed=42, device=None, **_ignore):
        self.rank, self.n_bags, self.epochs = rank, n_bags, epochs
        self.lr, self.weight_decay, self.aux_weight, self.seed = lr, weight_decay, aux_weight, seed
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))

    def fit(self, X: np.ndarray, y: np.ndarray, y_aux: np.ndarray, aux_w=None) -> "MultiTaskFM":
        """aux_w: None -> uniform scalar self.aux_weight; OR a per-row array (e.g. gated by |ghat|, d8's
        bite) to UN-STARVE THE ANTICIPATION — spend the aux only where d8 took a big bite (most starved)."""
        X = np.ascontiguousarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32).ravel()
        ya = np.asarray(y_aux, dtype=np.float32)
        if ya.ndim == 1:  # single aux -> [n,1]; SFV multi-objective stack -> [n, K]
            ya = ya[:, None]
        K = ya.shape[1]
        self._mu = X.mean(0, keepdims=True)
        s = X.std(0, keepdims=True)
        self._sd = np.where(s > 0, s, 1.0)
        Xz = ((X - self._mu) / self._sd).astype(np.float32)
        self._ym, ys = float(y.mean()), float(y.std())
        self._ys = ys if ys > 0 else 1.0
        a_m = ya.mean(0, keepdims=True)
        a_s = np.where(ya.std(0, keepdims=True) > 0, ya.std(0, keepdims=True), 1.0)
        Y = np.concatenate([((y - self._ym) / self._ys)[:, None], (ya - a_m) / a_s], 1).astype(np.float32)
        dev, n, d = self.device, len(Xz), Xz.shape[1]
        aw = (np.full(n, self.aux_weight, dtype=np.float32) if aux_w is None
              else np.asarray(aux_w, dtype=np.float32).ravel())
        Xt, Yt = torch.as_tensor(Xz, device=dev), torch.as_tensor(Y, device=dev)
        awt = torch.as_tensor(aw, device=dev)
        rng = np.random.default_rng(self.seed)
        self.models_ = []
        for b in range(self.n_bags):
            torch.manual_seed(self.seed + b)
            idx = torch.as_tensor(rng.integers(0, n, n), device=dev)
            m = _MultiHeadFMNet(d, 1 + K, self.rank).to(dev)
            opt = torch.optim.AdamW(m.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            xb, yb, awb = Xt[idx], Yt[idx], awt[idx]
            for _ in range(self.epochs):
                opt.zero_grad(set_to_none=True)
                pr = m(xb)
                loss = ((pr[:, 0] - yb[:, 0]) ** 2).mean()
                for kk in range(1, 1 + K):  # SFV multi-objective: per-row-weighted aux heads
                    loss = loss + (awb * (pr[:, kk] - yb[:, kk]) ** 2).mean()
                loss.backward()
                opt.step()
            self.models_.append(m)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        X = np.ascontiguousarray(X, dtype=np.float32)
        Xt = torch.as_tensor(((X - self._mu) / self._sd).astype(np.float32), device=self.device)
        with torch.no_grad():
            p = torch.stack([m(Xt)[:, 0] for m in self.models_], 0).mean(0)
        return (p.cpu().numpy() * self._ys + self._ym).astype(np.float64)
```

</details>

`_MultiHeadFMNet` is the shared FM interaction core both heads sit on — one factor matrix `V` builds the pairwise interactions, with a separate linear+bias per head, so the aux head's un-starving structure is carried in the very factors the primary (r2) head reads back:

<details>
<summary><code>src/models/regime_moe.py :: _MultiHeadFMNet</code></summary>

```python
class _MultiHeadFMNet(nn.Module):
    """Shared FM interaction core (factors V) + one linear+bias per head (the FM trick per head shares V)."""

    def __init__(self, d: int, heads: int, rank: int):
        super().__init__()
        self.V = nn.Parameter(torch.randn(d, rank) * 0.01)
        self.lin = nn.Linear(d, heads)
        self.b = nn.Parameter(torch.zeros(heads))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s = x @ self.V
        ss = (x * x) @ (self.V * self.V)
        return self.lin(x) + self.b + 0.5 * (s * s - ss).sum(1, keepdim=True)
```

</details>

---
## 3 · What this buys, and its honest bound

The CV harness turns "no global value works / it doesn't transfer" — the recurring shape of every dead lever in
this arc — from a *dead end* into the *design*: don't find the value, **fit it per step on purged val**. Per-step
CV pays for **cheap, low-variance, regime-dependent** hyperparameters (omega; the MTFM config aux_weight/gate/
rank/wd — validated, helps a hair beyond omega); it is **noise-dominated and HURTS** for **expensive,
high-variance, stable-optimum** ones (base-regularization CV: 0.19801 > 0.19645 fixed, alpha picks swinging
0.3↔30 — so base-alpha / d8-depth / EBM-cfg stay FIXED). All gains are **tiny** (weak-signal floor) but **real
and OOS-robust** — measured through the same gate that *correctly rejected* the architecture levers. The harness
is the home for context/attention ideas (fit on val, they can't self-disable) and, crucially, for the
**data-to-buy**: new information (auction imbalance / GEX / OFI / richer sentiment) is the only input not bounded
by the price-only floor, and it ships only when it clears this gate.